# Analogical reasoning

Humans solve novel problems by drawing on past experience - not by memorising specific answers, but by recognising that a new problem shares structural features with one already solved, and then adapting that prior solution to the new context. A city planner designing pedestrian flow might reach for crowd-dynamics research from biology. A startup founder thinking about distribution might look to how epidemics spread. The surface details are different; the underlying structure is the same.

Analogical reasoning makes this process explicit and systematic. Given a target problem, the system retrieves analogous problems from unrelated domains, evaluates which analogy shares the deepest structural similarity, creates explicit element-to-element correspondences (the "mapping"), and then uses those correspondences as a translation guide to adapt the analogous solution to the new context. Each step is transparent and inspectable, which distinguishes analogical reasoning from a model simply "generating a creative answer."

In this notebook we implement analogical reasoning step by step - one function at a time - building toward a complete four-phase pipeline: find, score, map, adapt.

In [1]:
import re
import os
from collections import namedtuple
from typing import List, Dict, Tuple

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0.7 encourages the model to explore cross-domain analogies
# that it might not surface at lower temperatures
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Representing an analogy
Each analogy retrieved from the LLM carries exactly two pieces of information: a description of the analogous problem from another domain, and a description of how that problem is typically solved. We need nothing else at retrieval time - the score and the mapping are computed in later steps and live outside this record.

A two-field `namedtuple` is the right structure here. Immutability is a genuine benefit: once we have retrieved an analogy from the LLM we should not need to modify it; the score and mapping that come later are products of comparison, not properties of the analogy itself. Keeping them separate prevents the kind of in-place mutation that made the original `@dataclass` implementation confusing to follow.

In [3]:
# Two fields — populated at retrieval time, never modified afterwards
Analogy = namedtuple('Analogy', ['problem', 'solution'])
# problem  — a concise description of the analogous problem from a different domain
#             e.g. "a city managing rush-hour traffic flow across multiple intersections"
# solution — how that analogous problem is typically solved
#             e.g. "adaptive signal timing based on real-time sensor feedback"

Scores and mappings are deliberately kept out of the `Analogy` record. A score is a *comparison result* between an analogy and a specific target problem - it belongs to the pair, not to the analogy alone. A mapping is an *analysis product* computed after the best analogy is selected.

## Finding analogous problems
The first and most creative step is retrieval - asking the model to identify problems from completely different domains that share structural characteristics with the target. The emphasis is on *structural* similarity, not surface similarity. "How does a restaurant manage peak-hour capacity?" is a surface analogy for a server load-balancing problem; "how does an ecosystem maintain stability when one species becomes dominant?" is a structural analogy - it shares the same underlying dynamics of resource competition, feedback loops, and equilibrium. The former gives us a slightly repackaged version of the same idea; the latter potentially gives us control-theory and diversity-based strategies that would not have come from thinking directly about servers.

In [4]:
def find_analogies(problem: str, num_analogies: int = 3) -> List[Analogy]:
    """Return num_analogies Analogy namedtuples from diverse domains."""
    # Emphasise structural over surface similarity explicitly in the prompt;
    # without this nudge, models tend to return close-domain variations
    prompt = (
        f"Given the following problem, identify {num_analogies} analogous problems from "
        f"completely different domains that share the same underlying structure — "
        f"similar constraints, dynamics, actors, or trade-offs.\n\n"
        f"Problem: {problem}\n\n"
        f"Avoid surface similarity (same industry or topic). Prefer deep structural matches "
        f"from fields like biology, engineering, logistics, history, or game theory.\n\n"
        f"Format each entry exactly as:\n"
        f"ANALOGY N:\n"
        f"PROBLEM: <analogous problem in one sentence>\n"
        f"SOLUTION: <how it is typically solved in one sentence>\n\n"
        f"Output only the formatted entries, nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    analogies = []
    current_problem = None
    current_solution = None

    for line in raw.splitlines():
        line = line.strip()
        if line.upper().startswith('ANALOGY'):
            # "ANALOGY N:" signals the start of a new entry; save the previous one first
            if current_problem and current_solution:
                analogies.append(Analogy(problem=current_problem, solution=current_solution))
            current_problem = None      # reset for the incoming entry
            current_solution = None
        elif line.upper().startswith('PROBLEM:'):
            current_problem = line.split(':', 1)[1].strip()   # text after "PROBLEM:"
        elif line.upper().startswith('SOLUTION:'):
            current_solution = line.split(':', 1)[1].strip()  # text after "SOLUTION:"

    # Capture the final entry — there is no trailing "ANALOGY N:" to flush it
    if current_problem and current_solution:
        analogies.append(Analogy(problem=current_problem, solution=current_solution))

    return analogies[:num_analogies]   # cap in case the model over-generates

The parser uses a small state machine: each `ANALOGY N:` header flushes the previous entry and resets both accumulators, while `PROBLEM:` and `SOLUTION:` lines populate the accumulators for the current entry. The final entry is flushed manually after the loop because there is no trailing header to trigger it. Splitting on the first colon only (`split(':', 1)`) ensures that colons inside the problem or solution text are preserved rather than discarded.

In [5]:
# Quick test: check that the function returns well-formed Analogy namedtuples
test_problem = "How can a small coffee shop compete with large chains?"
test_analogies = find_analogies(test_problem, num_analogies=3)

print(f"Problem: {test_problem}\n")
for i, a in enumerate(test_analogies, 1):
    print(f"Analogy {i}:")
    print(f"  Problem:  {a.problem}")
    print(f"  Solution: {a.solution}\n")

Problem: How can a small coffee shop compete with large chains?

Analogy 1:
  Problem:  How can a small, local farm thrive against industrial agriculture?
  Solution: They often focus on niche markets, organic practices, and community-supported agriculture to attract dedicated customers.

Analogy 2:
  Problem:  How can a small, independent bookstore survive in the age of online retailers?
  Solution: They commonly create a unique customer experience through author events, personalized recommendations, and local community engagement.

Analogy 3:
  Problem:  How can a small tech startup gain market share from established tech giants?
  Solution: They typically leverage innovation, agile development, and targeted marketing strategies to carve out a specific niche in the market.



## Scoring analogy fit
Not all analogies are equally useful. Some share deep structural patterns where the solution strategy transfers cleanly; others share only surface resemblance and break down as soon as we try to map the elements. The scoring step evaluates each analogy independently against three criteria: structural similarity (do the underlying dynamics match?), solution transferability (can the solving strategy actually be adapted?), and depth (is this a fundamental parallel or just a superficial one?). Scoring independently per analogy prevents the model from ranking them relative to each other — each analogy is evaluated against the target problem alone, not against the other candidates.

In [6]:
def score_analogy(problem: str, analogy: Analogy) -> float:
    """Score how well this analogy's structure maps to the target problem (0-10).

    Returns a float so scores from different analogies can be compared numerically.
    """
    prompt = (
        f"Evaluate how well the following analogy maps to the target problem.\n\n"
        f"Target problem: {problem}\n\n"
        f"Analogous problem: {analogy.problem}\n"
        f"How it's solved: {analogy.solution}\n\n"
        f"Score from 0 to 10 based on three criteria:\n"
        f"  - Structural similarity: shared constraints, actors, dynamics, or trade-offs\n"
        f"  - Solution transferability: how readily the solving approach adapts\n"
        f"  - Depth: fundamental parallel vs. only surface-level resemblance\n\n"
        f"Respond with exactly one line:\n"
        f"SCORE: <number from 0 to 10>"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    score = 5.0   # neutral fallback if the response cannot be parsed

    for line in raw.splitlines():
        if line.upper().startswith('SCORE:'):
            score_text = line.split(':', 1)[1].strip()   # text after "SCORE:"
            try:
                score = float(score_text)
                score = max(0.0, min(10.0, score))       # clamp to valid range
            except ValueError:
                # Model included surrounding text — extract the first number found
                match = re.search(r'\d+\.?\d*', score_text)
                if match:
                    score = max(0.0, min(10.0, float(match.group())))

    return score

The three-criterion rubric prevents the model from awarding high scores purely on topic proximity. "Solution transferability" is particularly important - an analogy that describes a fascinating parallel but whose solution is domain-specific and un-adaptable is less useful than a simpler analogy with a generic, transferable approach. The `re.search` fallback handles cases where the model returns `"SCORE: 7 out of 10"` or `"SCORE: approximately 8"` - the regex extracts the leading number regardless of surrounding text, and the subsequent clamp keeps it in range.

In [7]:
# Quick test: score each of the analogies we just retrieved
print(f"Scoring analogies for: {test_problem}\n")
test_scores = []
for i, analogy in enumerate(test_analogies, 1):
    score = score_analogy(test_problem, analogy)
    test_scores.append(score)
    print(f"  Analogy {i}: {score:.1f}/10  —  {analogy.problem[:70]}")

Scoring analogies for: How can a small coffee shop compete with large chains?

  Analogy 1: 8.0/10  —  How can a small, local farm thrive against industrial agriculture?
  Analogy 2: 9.0/10  —  How can a small, independent bookstore survive in the age of online re
  Analogy 3: 8.0/10  —  How can a small tech startup gain market share from established tech g


## Mapping structural elements
Selecting the best-scoring analogy is not enough on its own - we also need an explicit translation guide that says how concepts from the analogy correspond to concepts in the target problem. Without this, the adaptation step would have to infer correspondences on the fly, which tends to produce generic recommendations rather than genuinely translated insights. The mapping step makes the correspondence explicit: "the guild system in the analogy maps to the loyalty programme in the coffee shop problem", "the apprenticeship pathway maps to the barista training culture." These named correspondences are then handed directly to the adaptation prompt as a structured translation guide.

In [8]:
def map_analogy(problem: str, analogy: Analogy) -> Dict[str, str]:
    """Identify element-to-element correspondences between the analogy and the target problem.

    Returns a dict where each key is an analogy concept and each value is the
    corresponding concept in the target problem.
    """
    prompt = (
        f"Identify the structural correspondences between the following two problems.\n\n"
        f"Analogous problem: {analogy.problem}\n"
        f"How it's solved:   {analogy.solution}\n\n"
        f"Target problem: {problem}\n\n"
        f"List 3-5 key element correspondences. For each, name a concept from the analogy "
        f"and the concept it maps to in the target problem.\n\n"
        f"Format each correspondence as:\n"
        f"[analogy concept] -> [target problem concept]\n\n"
        f"Output only the correspondences, one per line."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    mapping = {}
    for line in raw.splitlines():
        if '->' in line:
            parts = line.split('->', 1)   # split on the first arrow only
            if len(parts) == 2:
                # Strip leading bullets, brackets, and whitespace from both sides
                key   = parts[0].strip().lstrip('-•*').strip().strip('[]')
                value = parts[1].strip().strip('[]')
                if key and value:          # skip any empty keys or values
                    mapping[key] = value

    return mapping

The prompt provides the analogy's solution text alongside the problem description - this matters because mapping is not just about the problem structure but about how the solution elements translate. Splitting on the first `->` only (`split('->', 1)`) handles cases where the value text itself contains an arrow. The `lstrip('-•*')` call handles models that prefix the list items with a bullet character, while `strip('[]')` removes the bracket notation suggested in the format example.

In [9]:
# Quick test: select the best analogy by score and create its mapping
best_idx = test_scores.index(max(test_scores))      # index of the highest score
best_analogy = test_analogies[best_idx]
best_score   = test_scores[best_idx]

print(f"Best analogy (A{best_idx + 1}, score {best_score:.1f}/10):")
print(f"  {best_analogy.problem}\n")

test_mapping = map_analogy(test_problem, best_analogy)
print("Structural mapping:")
for analogy_concept, problem_concept in test_mapping.items():
    print(f"  {analogy_concept}  →  {problem_concept}")

Best analogy (A2, score 9.0/10):
  How can a small, independent bookstore survive in the age of online retailers?

Structural mapping:
  unique customer experience  →  unique atmosphere or ambiance
  author events  →  local art showcases or live music
  personalized recommendations  →  customized drink options or specialty blends
  local community engagement  →  community events or partnerships with local businesses
  niche marketing  →  targeted promotions or loyalty programs


## Adapting the solution
The mapping is the translation dictionary; `adapt_solution` is the translator. It takes the best analogy, its solution, and the mapping, and asks the model to produce a concrete solution for the target problem by working through the correspondences systematically. The key instruction is to *produce an actionable solution for the target problem*, not to summarise the analogy. Without this constraint, models often produce a paragraph that describes the analogy in glowing terms and only gestures at the target problem at the end - which is the opposite of what we need. The mapping text is included verbatim in the prompt so the model has the translation guide in view while it generates the adaptation.

In [10]:
def adapt_solution(problem: str, analogy: Analogy, mapping: Dict[str, str]) -> str:
    """Translate the analogous solution to the target problem using the mapping.

    Returns a plain string containing the adapted solution.
    """
    # Build a readable summary of correspondences to include in the prompt
    mapping_text = "\n".join(
        f"  {analogy_concept} → {problem_concept}"
        for analogy_concept, problem_concept in mapping.items()
    )
    prompt = (
        f"Use the following analogy and its structural correspondences to solve the target problem.\n\n"
        f"Target problem: {problem}\n\n"
        f"Analogous problem: {analogy.problem}\n"
        f"Solution in the analogy domain: {analogy.solution}\n\n"
        f"Structural correspondences (analogy concept → target concept):\n{mapping_text}\n\n"
        f"Translate the analogous solution into a concrete, actionable solution for the target "
        f"problem by working through each correspondence. Do not summarise the analogy — "
        f"produce a solution that directly addresses the target problem. 3-5 sentences."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()   # plain string; caller decides how to display it

The `mapping_text` block is formatted `analogy concept → target concept` rather than the reverse, because the model is being asked to *read the analogy solution and translate it*. Presenting the correspondences in this direction gives the model a natural reading order: "whenever the analogy solution mentions X, interpret that as Y in the target problem." The "3-5 sentences" constraint prevents the model from producing a list of bullet points dressed up as a solution - it forces coherent prose that integrates the correspondences into a single argument.

In [11]:
# Quick test: adapt the best analogy's solution using the mapping
test_adapted = adapt_solution(test_problem, best_analogy, test_mapping)

print(f"Adapted solution for: {test_problem}\n")
print(test_adapted)

Adapted solution for: How can a small coffee shop compete with large chains?

To help the small coffee shop compete with large chains, it can create a unique atmosphere that reflects the local culture and invites customers to relax. Hosting local art showcases or live music events can draw in crowds and foster a sense of community. Offering customized drink options or specialty blends tailored to local tastes will set the shop apart from standard chain offerings. Additionally, the coffee shop can engage with the community through events and partnerships with local businesses, such as co-hosting workshops or promoting neighborhood events, to strengthen customer loyalty and attract new patrons. Implementing targeted promotions or a loyalty program can further encourage repeat visits and enhance customer relationships.


## The full analogical reasoning pipeline
With all four building blocks in place - `find_analogies`, `score_analogy`, `map_analogy`, and `adapt_solution` — the driver function chains them together and makes the selection logic explicit. Scores are kept in a separate list parallel to the analogies list, so `scores.index(max(scores))` directly gives the index of the best analogy without needing to sort or mutate the analogy objects. The function returns a plain dict so every intermediate result - all the analogies and their scores, the best analogy, the mapping, and the adapted solution - is inspectable after the call.

In [12]:
def analogical_reason(problem: str, num_analogies: int = 3) -> dict:
    """Full pipeline: find → score → select → map → adapt.

    Returns a dict with all intermediate data so callers can inspect any step.
    """
    print(f"Problem: {problem}\n")

    # Step 1 — Retrieve candidate analogies from diverse domains
    print(f"Step 1 — Finding {num_analogies} analogous problems...")
    analogies = find_analogies(problem, num_analogies)
    for i, a in enumerate(analogies, 1):
        print(f"  A{i}: {a.problem}")

    # Step 2 — Score each analogy independently against the target problem
    print(f"\nStep 2 — Scoring structural fit...")
    scores = []
    for i, analogy in enumerate(analogies, 1):
        print(f"  Scoring A{i}...")
        score = score_analogy(problem, analogy)
        scores.append(score)
        print(f"    {score:.1f}/10")

    # Select the best analogy by finding the maximum score in the parallel list
    best_idx   = scores.index(max(scores))    # index of highest-scoring analogy
    best       = analogies[best_idx]
    best_score = scores[best_idx]
    print(f"\n  Selected: A{best_idx + 1} — {best_score:.1f}/10")

    # Step 3 — Build explicit element correspondences for the best analogy
    print(f"\nStep 3 — Mapping structural elements...")
    mapping = map_analogy(problem, best)
    for k, v in mapping.items():
        print(f"  {k}  →  {v}")

    # Step 4 — Translate the analogous solution using the mapping as a guide
    print(f"\nStep 4 — Adapting solution...")
    adapted_solution = adapt_solution(problem, best, mapping)

    return {
        "problem":          problem,
        "analogies":        list(zip(analogies, scores)),   # [(Analogy, score), ...]
        "best_analogy":     best,
        "best_score":       best_score,
        "mapping":          mapping,
        "adapted_solution": adapted_solution,
    }

The `"analogies"` key stores a list of `(Analogy, score)` tuples - the parallel structure mirrors how scores were accumulated during step 2 and makes it easy for a caller to iterate over all analogies with their scores: `for analogy, score in result["analogies"]:`. The driver does not sort the analogies; the best is selected by index so its position in the original list is preserved for cross-referencing.

## Applying the full pipeline to a problem
Let's run the complete pipeline on a problem where cross-domain thinking genuinely helps: a startup struggling to acquire customers on a limited budget. Direct domain analysis tends to produce the same canonical advice (content marketing, referral programmes, cold outreach). Analogical reasoning may surface structural parallels from evolutionary biology, guerrilla warfare, or viral epidemiology - domains where resource-constrained actors have developed sophisticated strategies for expanding reach without scale.

In [13]:
problem = (
    "A startup has a good product but limited marketing budget and low brand awareness. "
    "How can they acquire new customers without significant spend?"
)

result = analogical_reason(problem, num_analogies=3)

print("\n" + "=" * 65)
print("RESULTS")
print("=" * 65)

print("\nAll analogies considered:\n")
for i, (analogy, score) in enumerate(result["analogies"], 1):
    print(f"  A{i} [{score:.1f}/10]: {analogy.problem}")

print(f"\nBest analogy ({result['best_score']:.1f}/10):")
print(f"  {result['best_analogy'].problem}")
print(f"  Solution: {result['best_analogy'].solution}")

print("\nStructural mapping:")
for k, v in result["mapping"].items():
    print(f"  {k}  →  {v}")

print("\nAdapted solution:\n")
print(result["adapted_solution"])

Problem: A startup has a good product but limited marketing budget and low brand awareness. How can they acquire new customers without significant spend?

Step 1 — Finding 3 analogous problems...
  A1: A small wildlife reserve wants to increase its animal population but has limited resources for breeding and habitat expansion.
  A2: A small town has a rich cultural heritage but lacks tourism funding and exposure to attract visitors.
  A3: A research team has a groundbreaking idea but limited funding and visibility in their field.

Step 2 — Scoring structural fit...
  Scoring A1...
    7.0/10
  Scoring A2...
    8.0/10
  Scoring A3...
    8.0/10

  Selected: A2 — 8.0/10

Step 3 — Mapping structural elements...
  cultural heritage  →  good product
  tourism funding  →  limited marketing budget
  grassroots marketing campaigns  →  low-cost marketing strategies
  local businesses  →  partnerships with other startups
  unique local events and attractions  →  unique selling propositions

Ste

The result dict's `"analogies"` key lets us compare all scored candidates after the fact - useful for understanding why the selected analogy was preferred. The display loop `for i, (analogy, score) in enumerate(result["analogies"], 1)` unpacks each `(Analogy, float)` tuple directly, reflecting the parallel list structure inside the driver.

**When to use analogical reasoning:** Novel problems where direct domain expertise provides only incremental improvements. Creative challenges where cross-domain transfer is the goal. Situations where you want a *traceable* argument for why a solution works - the mapping makes the reasoning auditable in a way that a direct LLM answer does not.

**Limitations to be aware of:** Analogy quality is sensitive to the retrieval prompt - models are prone to returning close-domain variations unless the prompt explicitly discourages them. The scoring step evaluates each analogy in isolation, so a genuinely outstanding analogy is scored correctly, but the absolute score values are not calibrated across runs. Each pipeline call costs `num_analogies + 3` LLM invocations; for three analogies that is six calls.

**Extending the pattern:** Add a second retrieval round that generates analogies specifically from domains the first round missed (e.g. "find analogies from biological systems only"). Use the mapping as a constraint on generation by asking the model to propose analogies that share at least three of a specified set of structural features. Combine analogical reasoning with hypothesis testing: generate an adapted solution via analogy, then generate competing hypotheses about why it would or would not work in the target context.